# segment

> Markdown becomes typed blocks that keep their file positions

In [1]:
#| default_exp segment

In [2]:
#| hide
from nbdev.showdoc import *

Rules read prose, but slopometer's input is markdown: READMEs, docstrings rendered from notebooks, PR text, chat responses. Segmentation turns a markdown document into typed blocks (heading, list item, prose paragraph) that remember where they came from, discards fenced code entirely, and blanks inline code so that lexical rules never fire on identifiers. Every downstream stage depends on the position bookkeeping here: a finding is only useful if it can point back at the exact file offset it came from.

In [3]:
#| export
from fastcore.utils import *
import mdhtml
from slopometer.core import *

In [4]:
from fastcore.test import *

A README-shaped fixture exercises everything segmentation must handle: headings, prose with inline code, a fenced block that must vanish, and a bullet list. It reappears throughout the later notebooks, and by the scoring notebook the meter reads it end to end.

In [5]:
SAMPLE = """# widget

A tool for widgets. Call `enhance()` to start.

```python
enhance(x)  # code is never scored
```

Install it:

    pip install widget

## Features

widget leverages a robust paradigm to deliver results.
It parses the config, then it writes the output.

- fast startup
- fast shutdown
- fast restarts
"""

Segmentation delegates block recognition to `mdhtml`, whose `blocks` function reports each top-level block's type and source line span. The mapping keeps three kinds. A `paragraph` becomes prose. A `heading` keeps its raw line. A `list` splits into one block per item line. Everything else drops: code blocks and fences, pipe tables, raw HTML, thematic breaks, and block quotes, since a quotation is not the author's prose. One class of junk needs handling before the parse. The `mdhtml` dialect renders out-of-subset tags such as `iframe` as literal text, on purpose, for paste safety. A scorer sees junk where the dialect sees text, and a regex therefore blanks every iframe span first, replacing each character with a space and keeping every newline. Offsets survive both steps. Each block records the offset its text starts at, which makes rebasing a finding onto file positions a single addition.


In [6]:
#| export
class Block:
    def __init__(self,
        kind, # 'heading', 'item', or 'prose'
        txt, # The block's text, sliced verbatim from the document after iframe blanking
        start, # Character offset of the block in the document
    ): store_attr()
    def __repr__(self): return f'<{self.kind} @{self.start}: {self.txt[:48]!r}>'

_iframe = re.compile(r'<iframe\b.*?</iframe\s*>', re.S | re.I)
_item = re.compile(r'^\s*(?:[-*+]|\d+[.)]) ')
_KINDS = dict(paragraph='prose', heading='heading')

def segment(doc):
    "Split markdown `doc` into typed `Block`s that keep their character offsets"
    doc = _iframe.sub(lambda m: re.sub(r'[^\n]', ' ', m.group()), doc)
    lines = doc.split('\n')
    offs, n = [], 0
    for l in lines:
        offs.append(n)
        n += len(l) + 1
    out = []
    for b in mdhtml.blocks(doc):
        seg = lines[b['start']:b['end']]
        if b['type'] == 'list':
            out += [Block('item', l, offs[b['start']+i]) for i,l in enumerate(seg) if _item.match(l)]
        elif b['type'] in _KINDS:
            txt = '\n'.join(seg).strip('\n')
            if txt.strip(): out.append(Block(_KINDS[b['type']], txt, offs[b['start']]))
    return L(out)

In [7]:
blocks = segment(SAMPLE)
blocks

[<heading @0: '# widget'>, <prose @10: 'A tool for widgets. Call `enhance()` to start.'>, <prose @108: 'Install it:'>, <heading @145: '## Features'>, <prose @158: 'widget leverages a robust paradigm to deliver re'>, <item @263: '- fast startup'>, <item @278: '- fast shutdown'>, <item @294: '- fast restarts'>]

The round-trip property is the contract every later stage leans on: a block's text is the document's own text at the block's offset. The fixture's fenced block is gone, and nothing else is.

In [8]:
for b in blocks: test_eq(SAMPLE[b.start:b.start+len(b.txt)], b.txt)
assert 'enhance(x)' not in ' '.join(blocks.attrgot('txt'))
assert 'pip install' not in ' '.join(blocks.attrgot('txt'))
len(blocks)

8

Inline code is exempt from every lexical rule: a function named `enhance` is not a style violation, and `write_docs` itself requires code symbols in backticks. Deleting the span would shift every offset after it, and that breaks the round-trip property. Filling the span with X characters removes the words and keeps the geometry, and the capitalized dummy token also preserves sentence boundaries: a space-blanked symbol at a sentence start would make the parser merge two sentences. Markdown link targets become spaces, since a URL is an address, not prose, and link text keeps the sentence readable on its own. A finding's offsets stay valid in the scrubbed text and in the original alike.

In [9]:
#| export
_icode = re.compile(r'`[^`\n]+`')
_ltarget = re.compile(r'\]\([^)\n]+\)')

def scrub(txt):
    "Replace inline code with X-fill and link targets with spaces, keeping every offset unchanged"
    txt = _icode.sub(lambda m: 'X'*len(m.group()), txt)
    return _ltarget.sub(lambda m: ' '*len(m.group()), txt)


In [10]:
scrubbed = scrub(blocks[1].txt)
scrubbed

'A tool for widgets. Call XXXXXXXXXXX to start.'

In [11]:
test_eq(len(scrubbed), len(blocks[1].txt))
test_eq(scrub('`cmd` runs fast.'), 'XXXXX runs fast.')
assert 'enhance' not in scrubbed
find_decoration(scrub('Works → fine `→ but code arrows are exempt →`'))


[[10] decoration (tell 24, decoration): '→' -> '->']

In [12]:
tricky = 'Intro prose stays.\n\n<iframe srcdoc="junk px solid">\nmore junk\n</iframe>\n\n| a | b |\n|---|---|\n| 1 | 2 |\n\n> a quoted voice, not the author\n\nReal prose after.\n'
tb = segment(tricky)
assert not any('junk' in b.txt or '|' in b.txt or 'quoted' in b.txt for b in tb)
tb

[<prose @0: 'Intro prose stays.'>, <prose @138: 'Real prose after.'>]

Two counters cover the whole document, and they are the only document-scale analysis in the meter: everything else stops at the paragraph. The over-structuring rule (tell 25) reads them in the scoring notebook.

In [13]:
#| export
def n_headings(blocks): return sum(1 for b in blocks if b.kind=='heading')
def prose_words(blocks): return sum(len(b.txt.split()) for b in blocks if b.kind=='prose')

In [14]:
n_headings(blocks), prose_words(blocks)

(2, 27)

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()